In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from scipy import stats

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [2]:
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [4]:
df.drop('deck',axis=1,inplace=True)

In [5]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


In [14]:
z_score = np.abs(stats.zscore(df['fare']))
filter_entries = np.where(z_score < 2.5)[0]
clean_data = df.iloc[filter_entries]


x = clean_data.drop('fare', axis=1)
y = clean_data['fare']

numeric_feature = ['age']
cat_feature = ['sex','sibsp', 'embarked','class','who','adult_male', 'parch','embark_town','alive','alone','survived','pclass',]

models = {
    'linear_model': LinearRegression(),
    'svm': SVR(),
    'neighbors': KNeighborsRegressor(),
    'tree': DecisionTreeRegressor(),
    'ensemble': RandomForestRegressor(),
    'xgboost': XGBRegressor(),
    'catboost': CatBoostRegressor(),
}

num_transformer = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='mean')),
    ('scale', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder()), 
])

preprocessing = ColumnTransformer(transformers=[
    ('num', num_transformer, numeric_feature),
    ('cat', cat_transformer, cat_feature)
])

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3,random_state=42)

for name, model in models.items():
    
    pipeline = Pipeline(steps=[
        ('preprocessing', preprocessing),
        ('model', model)
    ])

    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    print(f"mse of model :{name} is : {mean_squared_error(y_test,y_pred)}")
    print(f"r2 of model {name} is : {r2_score(y_test,y_pred)}")

mse of model :linear_model is : 426.8658822514257
r2 of model linear_model is : 0.599193959678795
mse of model :svm is : 801.7234986234042
r2 of model svm is : 0.2472211196151728
mse of model :neighbors is : 421.41391626914185
r2 of model neighbors is : 0.6043130872272413
mse of model :tree is : 415.4961137112972
r2 of model tree is : 0.6098696123777225
mse of model :ensemble is : 277.4986038650513
r2 of model ensemble is : 0.7394424777562734
mse of model :xgboost is : 322.16166977027984
r2 of model xgboost is : 0.6975060585239305
Learning rate set to 0.037847
0:	learn: 26.0248582	total: 1.42ms	remaining: 1.42s
1:	learn: 25.5334536	total: 2.69ms	remaining: 1.34s
2:	learn: 25.0318738	total: 4.66ms	remaining: 1.55s
3:	learn: 24.4955753	total: 6.54ms	remaining: 1.63s
4:	learn: 24.0200831	total: 8.25ms	remaining: 1.64s
5:	learn: 23.5664196	total: 9.4ms	remaining: 1.56s
6:	learn: 23.1598720	total: 11.3ms	remaining: 1.6s
7:	learn: 22.7119272	total: 13.6ms	remaining: 1.68s
8:	learn: 22.301106